In [ ]:
pip install selenium webdriver-manager beautifulsoup4 pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.3/486.3 kB 29.8 MB/s eta 0:00:00


In [ ]:
import time
import random
import logging
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

# Logging setup
logging.basicConfig(
    filename='scraper.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Rotating user-agents

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 8.0; Pixel 2 XL) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.127 Mobile Safari/537.36'
]

# Selenium setup with rotating proxies and user-agents
def create_driver(proxy=None, user_agent=None):
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # Run in headless mode
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    if proxy:
        chrome_options.add_argument(f'--proxy-server={proxy}')
    if user_agent:
        chrome_options.add_argument(f'--user-agent={user_agent}')
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    return driver


def fetch_page(url, proxy=None, user_agent=None):
    driver = create_driver(proxy, user_agent)
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, 'search'))
        )
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        return soup
    except Exception as e:
        logging.error(f"Failed to fetch page {url} with error: {e}")
        return None
    finally:
        driver.quit()


def get_title(soup):
    try:
        title = soup.find("span", attrs={"id": 'productTitle'})
        return title.text.strip()
    except AttributeError:
        return ""

def get_price(soup):
    try:
        price = soup.find("span", attrs={'id': 'priceblock_ourprice'}).text.strip()
    except AttributeError:
        try:
            price = soup.find("span", attrs={'id': 'priceblock_dealprice'}).text.strip()
        except:
            price = ""
    return price

def get_rating(soup):
    try:
        rating = soup.find("i", attrs={'class': 'a-icon a-icon-star a-star-4-5'}).text.strip()
    except AttributeError:
        try:
            rating = soup.find("span", attrs={'class': 'a-icon-alt'}).text.strip()
        except:
            rating = ""
    return rating

def get_review_count(soup):
    try:
        return soup.find("span", attrs={'id': 'acrCustomerReviewText'}).text.strip()
    except AttributeError:
        return ""

def get_availability(soup):
    try:
        available = soup.find("div", attrs={'id': 'availability'}).find("span").text.strip()
        return available
    except AttributeError:
        return "Not Available"


if __name__ == '__main__':
    base_url = "https://www.amazon.com/s?k=gaming+keyboard"

    # Rotating through proxies and user-agents
    proxy = random.choice(PROXIES)
    user_agent = random.choice(USER_AGENTS)


    soup = fetch_page(base_url, proxy=proxy, user_agent=user_agent)

    if soup:

        links = soup.find_all("a", attrs={'class': 'a-link-normal s-no-outline'})
        links_list = [f"https://www.amazon.com{link['href']}" for link in links if link.get('href')]


        data = {"title": [], "price": [], "rating": [], "reviews": [], "availability": []}


        for product_url in links_list[:10]:
            proxy = random.choice(PROXIES)
            user_agent = random.choice(USER_AGENTS)
            product_soup = fetch_page(product_url, proxy=proxy, user_agent=user_agent)

            if product_soup:
                data['title'].append(get_title(product_soup))
                data['price'].append(get_price(product_soup))
                data['rating'].append(get_rating(product_soup))
                data['reviews'].append(get_review_count(product_soup))
                data['availability'].append(get_availability(product_soup))
                time.sleep(random.uniform(3, 7))
            else:
                logging.warning(f"Failed to fetch product page: {product_url}")


        amazon_df = pd.DataFrame.from_dict(data)
        amazon_df['title'] = amazon_df['title'].replace('', np.nan)
        amazon_df = amazon_df.dropna(subset=['title'])
        amazon_df.to_csv("amazon_data.csv", header=True, index=False)
    else:
        logging.error("Failed to fetch the main Amazon search page.")


WebDriverException: Message: unknown error: cannot find Chrome binary
Stacktrace:
#0 0x5a142e40c4e3 <unknown>
#1 0x5a142e13bc76 <unknown>
#2 0x5a142e162757 <unknown>
#3 0x5a142e161029 <unknown>
#4 0x5a142e19fccc <unknown>
#5 0x5a142e19f47f <unknown>
#6 0x5a142e196de3 <unknown>
#7 0x5a142e16c2dd <unknown>
#8 0x5a142e16d34e <unknown>
#9 0x5a142e3cc3e4 <unknown>
#10 0x5a142e3d03d7 <unknown>
#11 0x5a142e3dab20 <unknown>
#12 0x5a142e3d1023 <unknown>
#13 0x5a142e39f1aa <unknown>
#14 0x5a142e3f56b8 <unknown>
#15 0x5a142e3f5847 <unknown>
#16 0x5a142e405243 <unknown>
#17 0x7ddc0530eac3 <unknown>
